In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import bce # type: ignore
import linear # type: ignore
import sigmoid # type: ignore

from common import repeat, test_model_grad # type: ignore

In [ ]:

class Per_Lin_Sig_BCE_Autograd(nn.Module):
    """
    A single-layer perceptron using `nn.Linear` for the affine transform, 
    `nn.Sigmoid` for the activation, and `nn.BCELoss` for training. 
    
    This is the highest-level, shortest, and fastest-to-implement version,
    relying fully on PyTorch to handle `forward` and `backward` passes.
    """
    
    def __init__(self, in_features, out_features):
        super().__init__()

        #
        # [Linear  + [BinaryCrossEntropyWithLogits = Sigmoid + BinaryCrossEntropy] 
        # is more numerically stable than [Linear] + [Sigmoid] + [BinaryCrossEntropy]
        #

        self.linear =  nn.Linear(in_features, out_features)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.model(x)
    
    def model(self, x):
        z = self.linear(x)
        p = self.sigmoid(z)
        return p

    def loss(self, p, y):
        return nn.BCELoss(reduction='mean')(p, y)

    def predict(self, x):
        with tr.no_grad():
            return (self.model(x) > 0.5).float()

In [ ]:

class Per_Lin_Sig_BCE_Backward(nn.Module):
    """
    A single-layer perceptron implemented using a fully custom autograd.Function 
    that defines both the forward pass and the analytical gradients for the `Linear`, 
    `Sigmoid`, and `Binary Cross Entropy` operations.
    
    The `forward` and `backward` passes for each operation are written manually, 
    but PyTorch’s autograd still orchestrates the overall gradient flow 
    by traversing the computation graph from the final loss.

    This is the mid-level variant: lower-level than the pure-PyTorch version, 
    yet still leveraging autograd to call the custom backward functions.
    """

    def __init__(self, in_features, out_features):
        super().__init__()

        #
        # [Linear Layer] + [BinaryCrossEntropyWithLogits = Sigmoid + BinaryCrossEntropy] 
        # is more numerically stable than [Linear] + [Sigmoid] + [BinaryCrossEntropy]
        #

        self.linear =  linear.Linear(in_features, out_features)
        self.sigmoid = sigmoid.Sigmoid()

    def model(self, x):
        z = self.linear(x)
        p = self.sigmoid(z)
        return p
    
    def loss(self, p, y):
        L = bce.BinaryCrossEntropy(reduction='mean')(p, y)
        return L

    def predict(self, x):
        with tr.no_grad():
            return (self.model(x) > 0.5).float()
        
    def forward(self, x):
        return self.model(x)

$$ z= xW + b $$
$$ p = \frac{1}{1+e^{-z}} $$
$$ L = -y \log(p) - (1-y) \log(1-p) $$

$$ \diamond \diamond \diamond $$

$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial W} = x^T \cdot (p-y) $$
$$ \frac{\partial L}{\partial b} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial b} = p-y $$


In [ ]:

class Per_Lin_Sig_BCE_Gradient_Function(autograd.Function):
    """
    A single-layer perceptron implemented using a fully custom autograd.Function 
    that defines both the forward pass and the analytical gradients for the `Linear`, 
    `Sigmoid`, and `Binary Cross Entropy` operations.
    
    PyTorch's autograd no longer computes derivatives for intermediate steps — instead, 
    the entire gradient flow is produced by the manually written backward method.
    
    This is the lowest-level variant, exposing the exact mechanics of gradient computation 
    and giving full control over how the model participates in PyTorch's autograd system.
    """

    @staticmethod
    def __model(x, W, b):
        z = linear.linear(x, W, b)
        p = sigmoid.sigmoid(z)
        return p
    
    @staticmethod
    def __loss(p, y):
        L = bce.binary_cross_entropy(p, y)
        return L

    @staticmethod
    def predict(x, W, b):
        with tr.no_grad():
            return (Per_Lin_Sig_BCE_Gradient_Function.__model(x, W, b) >= 0.5).float()

    @staticmethod
    def forward(ctx, x, W, b, y):
        p = Per_Lin_Sig_BCE_Gradient_Function.__model(x, W, b)
        L = Per_Lin_Sig_BCE_Gradient_Function.__loss(p, y)

        ctx.save_for_backward(x, W, y, p)

        return L
    
    @staticmethod
    def backward(ctx, dF_dL):
        x = ctx.saved_tensors[0]
        W = ctx.saved_tensors[1]
        y = ctx.saved_tensors[2]
        p = ctx.saved_tensors[3]
        
        S = x.shape[0]  # Samples
        FI = x.shape[1] # Features In
        FO = W.shape[1] # Features Out

        assert x.shape == (S, FI)
        assert W.shape == (FI, FO)
        assert p.shape == (S, FO)
        assert y.shape == (S, FO)
        
        dL_dz = (p - y)

        # Average over all elements
        N = S * FO
        dL_dz /= N

        # For this example dF_dL == 1 but we include it 
        # to show how the chain rule works in general.
        dL_dz *= dF_dL
        assert dL_dz.shape == (S, FO)
       
        # (FI, FO) = (S, FI).T @ (S, FO)
        dL_dW = x.t() @ dL_dz
        assert dL_dW.shape == (FI, FO)

        dL_db = dL_dz.sum(dim=0)
        assert dL_db.shape == (FO,)

        # Return values for each `forward` argument, in the same order.
        # Only `W` and `b` are parameters and require gradients.
        return (None, dL_dW, dL_db, None)
    

class Per_Lin_Sig_BCE_Gradient(nn.Module):
    """
    A single-layer linear perceptron implemented using a fully custom autograd.Function.
    """

    # In general the perceptron model is separable from the loss function, 
    # but since the total gradient is calculated manually, the loss function 
    # is an integrated part of it. This is helper for testing to indicate that 
    # the `forward` method expects both, `x` and `y`.
    CUSTOM_GRAD=True

    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(tr.randn(in_features, out_features, dtype=tr.float32))
        self.bias = nn.Parameter(tr.randn(out_features, dtype=tr.float32))

    def model(self, x):
        with tr.no_grad():
            return Per_Lin_Sig_BCE_Gradient_Function.__model(x, self.weight, self.bias)
    
    def loss(self, p, y):
        with tr.no_grad():
            return Per_Lin_Sig_BCE_Gradient_Function.__loss(p, y)

    def predict(self, x):
        with tr.no_grad():
            return Per_Lin_Sig_BCE_Gradient_Function.predict(x, self.weight, self.bias)
        
    def forward(self, x, y):
        assert x.shape[1] == self.weight.shape[0]
        assert y.shape[1] == self.weight.shape[1]
                
        return Per_Lin_Sig_BCE_Gradient_Function.apply(x, self.weight, self.bias, y)
    

def test_Per_Lin_Sig_BCE_Gradient_checkgrad():
    S, FI, FO = 4, 3, 2

    x = tr.randn(S, FI, dtype=tr.double, requires_grad=False)
    W = tr.randn(FI, FO, dtype=tr.double, requires_grad=True)
    b = tr.randn(FO, dtype=tr.double, requires_grad=True)
    y = tr.randn(S, FO, dtype=tr.double, requires_grad=False)
    inputs = (x, W, b, y)

    assert autograd.gradcheck(Per_Lin_Sig_BCE_Gradient_Function.apply, inputs, eps=0.0001, atol=0.0001)


if __name__ == "__main__":
    test_Per_Lin_Sig_BCE_Gradient_checkgrad()


In [ ]:
def test_logical_fn(model, bool_fn, samples=100, epochs=500, lr=0.1):
    x = (tr.randn(samples, 2, dtype=tr.float32) > 0).float()
    y = bool_fn(x[:, 0], x[:, 1]).float().unsqueeze(1)
    return test_model_grad(model, x, y, epochs, lr=lr)


if __name__ == "__main__":
    #
    # The `and`, `or`, and `nand` functions are linearly separable, so the perceptron should be able to learn them.
    #

    E = 500
    
    logical_and = tr.logical_and  
    assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Autograd(2, 1), logical_and, epochs=E)) >= 0.98
    assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Backward(2, 1), logical_and, epochs=E)) >= 0.98
    assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Gradient(2, 1), logical_and, epochs=E)) >= 0.98

    logical_or = tr.logical_or
    assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Autograd(2, 1), logical_or, epochs=E)) >= 0.98
    assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Backward(2, 1), logical_or, epochs=E)) >= 0.98
    assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Gradient(2, 1), logical_or, epochs=E)) >= 0.98

    logical_nand = lambda x1, x2: tr.logical_not(tr.logical_and(x1, x2))
    assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Autograd(2, 1), logical_nand, epochs=E)) >= 0.98
    assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Backward(2, 1), logical_nand, epochs=E)) >= 0.98
    assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Gradient(2, 1), logical_nand, epochs=E)) >= 0.98
    
    #
    # The `xor` function is not linearly separable, so the perceptron should not be able to learn it.
    #

    for E in [1000, 2000, 3000]:
        assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Autograd(2, 1), tr.logical_xor, epochs=E)) <= 0.75
        assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Backward(2, 1), tr.logical_xor, epochs=E)) <= 0.75
        assert repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Gradient(2, 1), tr.logical_xor, epochs=E)) <= 0.75
